In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

In [ ]:
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import jax.numpy as jnp

from rhmag.utils.pretest_evaluation import create_multilevel_df
from rhmag.data_management import FINAL_MATERIALS, MaterialSet, DataSet
from rhmag.utils.data_plotting import plot_sequence_prediction, plot_hysteresis_prediction
from rhmag.utils.model_evaluation import reconstruct_model_from_file, plot_model_frequency_sweep, evaluate_cross_validation, get_exp_ids
from rhmag.utils.final_data_evaluation import (
    FINAL_MATERIALS,
    TestSet,
    ResultSet,
    predict_test_scenarios,
    validate_result_set,
    visualize_result_set,
    evaluate_test_scenarios,
    update_pareto_df,
    get_exp_ids_per_material,
)
from rhmag.model_setup import setup_normalizer, setup_dataset

In [ ]:
test_data_per_material = {material_name: TestSet.from_material_name(material_name) for material_name in FINAL_MATERIALS}

#### Quantitative Comparison:

In [ ]:
exp_ids_per_material=get_exp_ids_per_material(
    model_type="GRU8",
    exp_name="final-f32",
)

not_loadable = {material_name: [] for material_name in exp_ids_per_material.keys()}
loadable = {material_name: [] for material_name in exp_ids_per_material.keys()}

for material_name, exp_ids in exp_ids_per_material.items():
    for exp_id in exp_ids:
        try:
            reconstruct_model_from_file(exp_id)
            loadable[material_name].append(exp_id)
        except KeyError:
            print(exp_id, "could not be loaded.")
            not_loadable[material_name].append(exp_id)


full_features_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=loadable,
    test_data_per_material=test_data_per_material,
)

In [ ]:
reduced_features_results_df = update_pareto_df(
    pareto_results_path=None,
    exp_ids_per_material=get_exp_ids_per_material(
        model_type="GRU8",
        exp_name="ablation-default-f32",
    ),
    test_data_per_material=test_data_per_material,
)

In [ ]:
# compare the results

display("full_features_results_df:", full_features_results_df.groupby("material").mean(numeric_only=True))
display("reduced_features_results_df:", reduced_features_results_df.groupby("material").mean(numeric_only=True))

In [ ]:
import pandas as pd
import seaborn as sns

In [ ]:
# combine into a single df:

reduced_features_results_df["init_type"] = "reduced features"
full_features_results_df["init_type"] = "full features"


full_results_df = pd.concat([reduced_features_results_df, full_features_results_df], ignore_index=True)

In [ ]:
import matplotlib as mpl
from matplotlib import rc
from matplotlib.ticker import ScalarFormatter, StrMethodFormatter, LogLocator

rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}"

In [ ]:
def plot_ablation_comparison(full_results_df):
    fig, axs = plt.subplots(1, 4, figsize=(7.167, 7.167 / 3), constrained_layout=True)
    metric_label_map = {
        "sre_avg": r"$\mathrm{SRE}_{\mathrm{avg}}$", 
        "sre_95th": r"$\mathrm{SRE}_{95\mathrm{-th}}$",
        "nere_avg": r"$\mathrm{NERE}_{\mathrm{avg}}$",
        "nere_95th": r"$\mathrm{NERE}_{95\mathrm{-th}}$",
    }
    
    for idx, (ax, metric) in enumerate(zip(axs, ["sre_avg", "sre_95th", "nere_avg", "nere_95th"])):
        sns.stripplot(
            data=full_results_df,
            x="material",
            y=metric,
            hue="init_type",
            legend=True if idx==0 else False,
            ax=ax,
            alpha=0.7,
        )
        ax.set_ylabel(metric_label_map[metric])
        ax.set_yscale('log')
        ax.tick_params(which="both", axis="y", direction="in")
        ax.tick_params(which="both", axis="x", direction="in")
        ax.yaxis.set_major_formatter(StrMethodFormatter('{x:.3f}'))
        ax.yaxis.set_minor_formatter(StrMethodFormatter('{x:.3f}'))        
        ax.grid(True, which="both", alpha=0.3)

    handles, labels = axs[0].get_legend_handles_labels()

    print(handles, labels)
    fig.legend(
        handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2
    )
    axs[0].legend().remove()

    return fig, axs

In [ ]:
fig, axs = plot_ablation_comparison(full_results_df)

# plt.savefig("ablation_study_init_quantitative.png", bbox_inches="tight", dpi=300)
plt.savefig("ablation_study_features_quantitative.pdf", bbox_inches="tight")
plt.show()